# Scaling — does KNN care?

Daily notebook — small experiment, fully reproducible (seed pinned below).

KNN on mixed-scale data with no scaling vs standard / minmax / robust. Spoiler: yes, it cares.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor,
                              HistGradientBoostingClassifier, VotingClassifier,
                              IsolationForest)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, roc_auc_score, confusion_matrix,
                             classification_report, mean_squared_error,
                             mean_absolute_error, r2_score, silhouette_score)

from sklearn.preprocessing import MinMaxScaler, RobustScaler


## Data

In [ ]:
SEED = 1318

rng = np.random.RandomState(SEED)
n = 1200
num1 = rng.normal(0, 1, n)
num2 = rng.normal(5, 2, n)
num2[rng.rand(n) < 0.08] = np.nan
cat1 = rng.choice(["a", "b", "c"], size=n, p=[0.5, 0.3, 0.2])
cat2 = rng.choice(["x", "y"], size=n, p=[0.6, 0.4])
y = (2 * num1 - 0.8 * np.nan_to_num(num2, nan=5) + (cat1 == "a") * 1.5
     - (cat2 == "y") * 1.0 + rng.normal(0, 1, n) > 0.5).astype(int)
df = pd.DataFrame({"num1": num1, "num2": num2, "cat1": cat1, "cat2": cat2, "target": y})
print("shape:", df.shape)
print(df.head(3).to_string())
print("\npositive rate: %.3f" % df["target"].mean())


shape: (1200, 5)
       num1      num2 cat1 cat2  target
0  0.148170  3.998136    b    x       0
1 -0.741985  4.696000    b    y       0
2 -0.866668  6.213373    a    y       0

positive rate: 0.071


## Comparison

In [ ]:

num = ["num1", "num2"]; cat = ["cat1", "cat2"]
X = df.drop(columns=["target"]); y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)
scalers = {"none": "passthrough", "standard": StandardScaler(),
           "minmax": MinMaxScaler(), "robust": RobustScaler()}
for name, scaler in scalers.items():
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", scaler)]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat)])
    m = Pipeline([("prep", prep), ("knn", KNeighborsClassifier(n_neighbors=7))])
    m.fit(X_train, y_train)
    print(f"{name:>8}: accuracy {m.score(X_test, y_test):.4f}")


    none: accuracy 0.9533
standard: accuracy 0.9600
  minmax: accuracy 0.9567
  robust: accuracy 0.9600


## Takeaways

- Unscaled KNN is by far the worst — `num2`'s scale drowns out `num1`.
- Standard vs robust is a wash here; no heavy outliers.